# Aggregating 1 Minute OHLCV bars into 5 minutes bars using Nautilus Trader's in-built "Bar Aggregators"

In [ ]:
import pandas as pd
from nautilus_trader.persistence.catalog import ParquetDataCatalog 
from nautilus_trader.model.data import Bar, BarType , BarAggregation, BarSpecification
from nautilus_trader.model.enums import PriceType
from nautilus_trader.data.aggregation import TimeBarAggregator
from nautilus_trader.common.component import LiveClock
from nautilus_trader.model.objects import Currency
from nautilus_trader.model.objects import Money
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.identifiers import Venue

from nautilus_trader.model.instruments.crypto_perpetual import CryptoPerpetual

from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Price, Quantity
from nautilus_trader.core.datetime import millis_to_nanos
from decimal import Decimal

CSV_PATH = "./data/sample_1min_ohlcv_2025_unix_timestamp.csv"
CATALOG_PATH = "./catalog" 
INSTRUMENT_ID = InstrumentId.from_str("BTCUSDT.BINANCE")

df = pd.read_csv(CSV_PATH)
catalog = ParquetDataCatalog(CATALOG_PATH)
df.head()

# Creating Bar Specification & Bar Type

In [ ]:
bar_spec = BarSpecification(
    step=1,
    aggregation=BarAggregation.MINUTE,
    price_type=PriceType.LAST,
)

bar_type = BarType(
    instrument_id = INSTRUMENT_ID,
    bar_spec = bar_spec,
)



In [ ]:
bars = []

for _, row in df.iterrows():

    ts_event = millis_to_nanos(int(row["timestamp"]))

    bar = Bar(
    bar_type=bar_type,

    open=Price.from_str(str(row["open"])),
    high=Price.from_str(str(row["high"])),
    low=Price.from_str(str(row["low"])),
    close=Price.from_str(str(row["close"])),

    volume=Quantity.from_int(int(row["volume"])),

    ts_event=ts_event,
    ts_init=ts_event,
    )

    bars.append(bar)

print(f"Created {len(bars)} bars","\n")

# Writing the bars to ParquetDataCatalog

In [ ]:
catalog = ParquetDataCatalog(CATALOG_PATH)
catalog.write_data(bars)
print("Written to ParquetCatalog")

# Aggregation Pipeline
- Now we will work with the earlier created ParquetDataCatalog.
- We will try to generate the following timestamps:
- 5 Min, 30 Min, 1 Hour, 2 Hour, 1 day, 1 Week, 1 Month

In [ ]:
catalog = ParquetDataCatalog(CATALOG_PATH)

bar_spec_5m = BarSpecification(
    step = 5,
    aggregation = BarAggregation.MINUTE,
    price_type = PriceType.LAST
)

bar_type_5m = BarType(
    bar_spec = bar_spec_5m,
    instrument_id = INSTRUMENT_ID
)


rows = []

for bar in bars:
    rows.append({
        "timestamp": pd.to_datetime(bar.ts_event, unit="ns"),
        "open": float(bar.open),
        "high": float(bar.high),
        "low": float(bar.low),
        "close": float(bar.close),
        "volume": float(bar.volume),
    })

df = pd.DataFrame(rows)
df.set_index("timestamp", inplace=True)

df_5m = df.resample("5min").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
})

df_5m.dropna(inplace=True)


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1748 entries, 2025-01-01 09:15:00 to 2025-01-31 15:30:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    1748 non-null   float64
 1   high    1748 non-null   float64
 2   low     1748 non-null   float64
 3   close   1748 non-null   float64
 4   volume  1748 non-null   float64
dtypes: float64(5)
memory usage: 81.9 KB


# Creating an Instrument
- Sometimes instruments are not stored inside ParquetDataCatalogs
- In this case, the Nautilus components that depend on Instrument Metadata (tick_size, qty_size, symbol etc.) will not work properly.

In [23]:
def create_instrument():
    """Creates an instrument and writes it 
        in the ParquetData Catalog.
    """
    instrument = CryptoPerpetual(
        instrument_id = INSTRUMENT_ID,
        raw_symbol = Symbol("BTCUSDT"),
        base_currency = Currency.from_str("BTC"),
        quote_currency = Currency.from_str("USDT"),
        settlement_currency = Currency.from_str("USDT"),

        is_inverse = False,
        price_precision = 2,
        size_precision = 3,

        price_increment = Price.from_str("0.01"),

        size_increment = Quantity.from_str("0.001"),

        lot_size = Quantity.from_str("0.001"),
        max_quantity = Quantity.from_str("1000"),
        min_quantity = Quantity.from_str("0.001"),
        
        ts_event = 0,
        ts_init = 0,
    )

    return instrument 

instrument = create_instrument()
catalog.write_data([instrument])

# Using the Data from Data Catalog, along with some custom Clock objects to create the Aggregator Instance 

In [34]:
bars = sorted(bars,key = lambda x: x.ts_event)

print("Bars count:", len(bars))

print("Input bar type:", bars[0].bar_type)
print("Target bar type:", bar_type_5m)

print("Instrument in bars:", bars[0].bar_type.instrument_id)
print("Instrument object:", instrument.id)

print("First timestamp:", bars[0].ts_event)
print("Second timestamp:", bars[1].ts_event)

print("Timestamp diff:", bars[1].ts_event - bars[0].ts_event)

aggregated_bars = []


def handler(bar: Bar)-> None:
    aggregated_bars.append(bar)

clock = LiveClock()
instrument = catalog.instruments(instrument_ids = [INSTRUMENT_ID])[0]
print("Instrument = ", instrument)

# Nautilus Aggregator acts as even-driven (converts each bar in a stream of bars to specifiec bar_type) 
aggregator = TimeBarAggregator(
    bar_type = bar_type_5m,
    instrument = instrument,
    clock = clock,
    handler = handler
)

for bar in bars:
    aggregator.handle_bar(bar)

print(f"Length of generated aggregated bars = {len(aggregated_bars)}")



Bars count: 8648
Input bar type: BTCUSDT.BINANCE-1-MINUTE-LAST-EXTERNAL
Target bar type: BTCUSDT.BINANCE-5-MINUTE-LAST-EXTERNAL
Instrument in bars: BTCUSDT.BINANCE
Instrument object: BTCUSDT.BINANCE
First timestamp: 1735722900000000000
Second timestamp: 1735722960000000000
Timestamp diff: 60000000000
Instrument =  CryptoPerpetual(id=BTCUSDT.BINANCE, raw_symbol=BTCUSDT, asset_class=CRYPTOCURRENCY, instrument_class=SWAP, quote_currency=USDT, is_inverse=False, price_precision=2, price_increment=0.01, size_precision=3, size_increment=0.001, multiplier=1, lot_size=1, margin_init=0, margin_maint=0, maker_fee=0, taker_fee=0, info=None)
Length of generated aggregated bars = 0
